In [1]:
import random
import torch

In [2]:
random.seed(67)
torch.manual_seed(67)

Let's start with a batch of xs with batch_size = 2, seq_len = 3, d_emb = 4.  So that's a (2, 3, 4) tensor:

In [3]:
batch_size = 2
seq_len = 3
d_emb = 4

In [4]:
xs = torch.rand((batch_size, seq_len, d_emb))

In [5]:
xs.shape

torch.Size([2, 3, 4])

Now, let's say we have seven experts.  We get weights with a linear layer mapping from d_emb to num_experts.

In [6]:
num_experts = 7

In [7]:
router = torch.nn.Linear(d_emb, num_experts, bias=False)

In [8]:
routing_logits = router(xs)

In [9]:
routing_logits.shape

torch.Size([2, 3, 7])

As expected, it's (batch_size, seq_len, num_experts)

In [10]:
routing_logits

tensor([[[ 0.1627, -0.3312,  0.7548,  0.3239, -0.4476,  0.4251, -0.3319],
         [ 0.0934, -0.4510,  0.9410,  0.3338, -0.4869,  0.5465, -0.2595],
         [ 0.2307, -0.2847,  0.7082,  0.1746, -0.3453,  0.3372, -0.4413]],

        [[ 0.0398, -0.2237,  0.6263,  0.2902, -0.3324,  0.3782, -0.1889],
         [ 0.0599, -0.2611,  0.2460,  0.0296, -0.1585,  0.1370, -0.0216],
         [ 0.0427, -0.2502,  0.5480,  0.1718, -0.2585,  0.3135, -0.1475]]],
       grad_fn=<UnsafeViewBackward0>)

Let's have five active -- very artificial, but it means that all of our axis sizes are different which should make things easier to compare.

In [11]:
num_active_experts = 5

In [12]:
top_k_values, top_k_indices = torch.topk(
    routing_logits, 
    k=num_active_experts,
    dim=-1
)

In [13]:
top_k_values.shape

torch.Size([2, 3, 5])

Now, previously I identified the lowest top-k expert logit for each of the input samples, then set all values below that to -inf, then ran it through softmax.  In our example here, we're routing to our top five of seven experts. Imagine that for one sample, the expert logits were `(1, 2, 3, 4, 5, 6, 7)`.  We get a smallest top-5 logit of 3, so we overwrite that: `(-inf, -inf, 3, 4, 5, 6, 7)`,  Running that through softmax gives us appropriate weights, with zeros for the first two and probabilities for the others.

But there's a problem.  Imagine that for another sample, the expert logits were `(1, 2, 2, 3, 4, 5, 6)`.  The smallest top-5 logit is 2, so if we overwrite smaller values than that with `-inf` we get `(-inf, 2, 2, 3, 4, 5, 6)`.  We now have six active experts rather than 5!

Now, `torch.topk` will do a tie-break for us -- if there happens to be a match, it will just randomly select one of them, and will return exactly $k$ in the `top_k_indices`.  So that's actually what we need to use.


In [14]:
top_k_indices

tensor([[[2, 5, 3, 0, 1],
         [2, 5, 3, 0, 6],
         [2, 5, 0, 3, 1]],

        [[2, 5, 3, 0, 6],
         [2, 5, 0, 3, 6],
         [2, 5, 3, 0, 6]]])

Let's start with an all-`-inf` tensor the same size as `routing_logits`

In [15]:
top_k_routing_logits = torch.full_like(routing_logits, -torch.inf)

In [16]:
top_k_routing_logits

tensor([[[-inf, -inf, -inf, -inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf, -inf, -inf, -inf]],

        [[-inf, -inf, -inf, -inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf, -inf, -inf, -inf]]])

In [17]:
top_k_routing_logits.shape

torch.Size([2, 3, 7])

In [18]:
top_k_indices.shape

torch.Size([2, 3, 5])

Now we can use the `scatter_` method on that tensor.  The underscore, as usual, is to say "do this in place".  It takes a parameter that is the dimension to do this on (-1 in our case), the tensor of indices, and the tensor of the values to copy in.  We want to copy the top-k values into our array in the top-k positions, So:

In [19]:
top_k_routing_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_values) 

tensor([[[ 0.1627, -0.3312,  0.7548,  0.3239,    -inf,  0.4251,    -inf],
         [ 0.0934,    -inf,  0.9410,  0.3338,    -inf,  0.5465, -0.2595],
         [ 0.2307, -0.2847,  0.7082,  0.1746,    -inf,  0.3372,    -inf]],

        [[ 0.0398,    -inf,  0.6263,  0.2902,    -inf,  0.3782, -0.1889],
         [ 0.0599,    -inf,  0.2460,  0.0296,    -inf,  0.1370, -0.0216],
         [ 0.0427,    -inf,  0.5480,  0.1718,    -inf,  0.3135, -0.1475]]],
       grad_fn=<ScatterBackward0>)

Quick sanity check -- we can see if the values in the original un-masked tensor match up, with just the smallest in each displayed row replaced with `-inf`

In [20]:
routing_logits

tensor([[[ 0.1627, -0.3312,  0.7548,  0.3239, -0.4476,  0.4251, -0.3319],
         [ 0.0934, -0.4510,  0.9410,  0.3338, -0.4869,  0.5465, -0.2595],
         [ 0.2307, -0.2847,  0.7082,  0.1746, -0.3453,  0.3372, -0.4413]],

        [[ 0.0398, -0.2237,  0.6263,  0.2902, -0.3324,  0.3782, -0.1889],
         [ 0.0599, -0.2611,  0.2460,  0.0296, -0.1585,  0.1370, -0.0216],
         [ 0.0427, -0.2502,  0.5480,  0.1718, -0.2585,  0.3135, -0.1475]]],
       grad_fn=<UnsafeViewBackward0>)

They are!  So now we can softmax the result to get weights

In [21]:
expert_weights = torch.softmax(top_k_routing_logits, dim=-1)

In [22]:
expert_weights.shape

torch.Size([2, 3, 7])

In [23]:
expert_weights

tensor([[[0.1697, 0.1036, 0.3068, 0.1994, 0.0000, 0.2206, 0.0000],
         [0.1453, 0.0000, 0.3392, 0.1848, 0.0000, 0.2286, 0.1021],
         [0.1899, 0.1134, 0.3060, 0.1795, 0.0000, 0.2112, 0.0000]],

        [[0.1592, 0.0000, 0.2862, 0.2045, 0.0000, 0.2233, 0.1267],
         [0.1932, 0.0000, 0.2327, 0.1874, 0.0000, 0.2087, 0.1781],
         [0.1685, 0.0000, 0.2794, 0.1918, 0.0000, 0.2210, 0.1394]]],
       grad_fn=<SoftmaxBackward0>)

OK, let's see how we'd work with expert 1.  It's the second column in each of those batch items above.  It has two active experts.

In [24]:
expert_ix = 1

Firstly, let's see which items should go through it?

In [25]:
this_expert_mask = expert_weights[:, :, expert_ix] > 0

In [26]:
this_expert_mask.shape

torch.Size([2, 3])

So that's (batch_size, seq_len) and shows which tokens should go through expert 1.

In [27]:
this_expert_mask

tensor([[ True, False,  True],
        [False, False, False]])

So that matches the right sequences

In [28]:
xs

tensor([[[0.7380, 0.3646, 0.8673, 0.5464],
         [0.5885, 0.9785, 0.8875, 0.9903],
         [0.3595, 0.3245, 0.8871, 0.9991]],

        [[0.5664, 0.5706, 0.7217, 0.2157],
         [0.0415, 0.2903, 0.0052, 0.8127],
         [0.2556, 0.6512, 0.5186, 0.6332]]])

In [29]:
xs[this_expert_mask]

tensor([[0.7380, 0.3646, 0.8673, 0.5464],
        [0.3595, 0.3245, 0.8871, 0.9991]])

...and that is indeed the batch that we want to run through -- the first and the third tokens from the first sequence in the batch.  Let's fake up the results to that.  In our case, our results are the same shape as the inputs, so let's make it all ones.

In [30]:
this_expert_results = torch.ones_like(xs[this_expert_mask])

In [31]:
this_expert_results.shape

torch.Size([2, 4])

So that's shaped (number_of_tokens_for_this_expert, d_emb)

In [32]:
this_expert_results

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])

Now, we want to multiply those by the appropriate weights (the non-zero values in the two second columns of the `expert_weights` tensor above).  And then we'll want to aggregate them with other results from other experts.  A neat way of doing that is to start of with all-zero results in the same shape as our inputs (again, we're relying on inputs and outputs having the same shape) and then to add these, weighted, into the right slots in there.  Let's do that step by step.

In [33]:
all_outputs = torch.zeros_like(xs)

Let's get the weights we want to multiply them by, in the same shape as our results.

In [34]:
this_expert_weights = expert_weights[this_expert_mask, expert_ix].unsqueeze(1)

In [35]:
this_expert_weights.shape

torch.Size([2, 1])

In [36]:
this_expert_weights

tensor([[0.1036],
        [0.1134]], grad_fn=<UnsqueezeBackward0>)

In [37]:
this_expert_results * this_expert_weights

tensor([[0.1036, 0.1036, 0.1036, 0.1036],
        [0.1134, 0.1134, 0.1134, 0.1134]], grad_fn=<MulBackward0>)

So, we have our output results, each one weighted by the weight that the specified token had for the specified expert.  Now we can add them in to the outputs

In [38]:
all_outputs[this_expert_mask] += this_expert_results * this_expert_weights

In [39]:
all_outputs

tensor([[[0.1036, 0.1036, 0.1036, 0.1036],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.1134, 0.1134, 0.1134, 0.1134]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]]], grad_fn=<IndexPutBackward0>)

And we're done!  Iterate that over the other experts and we'll get our results.